In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
import palettable
import numpy as np
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from datetime import datetime
import os
from machine_intelligence.utilities.viz_setup import figure_setup_grid, VariableNames

from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
import matplotlib.ticker as ticker
import random

from machine_intelligence.utilities.data_handling import load_parquet_table
from machine_intelligence.models.GatedTransformer import GatedTransformer
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule



In [ ]:

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)
# model_path = project_root / Path("experiment_logs/gtn-debug/version_118/checkpoints/epoch=5-step=186.ckpt")

model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final-3/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(model_path, map_location="cpu")



In [ ]:

wandb.init(mode="disabled")

os.environ['PYTHONHASHSEED'] = str(42)
lit.seed_everything(42, workers=True)
torch.use_deterministic_algorithms(True)
torch.cuda.manual_seed_all(42)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_prev_0.3_imp_none"])

    print(cfg)

In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)
sepsis_pred_data.setup(stage="fit")

In [ ]:
df_train = sepsis_pred_data.df_train
df_val = sepsis_pred_data.df_val

In [ ]:
df_train.head()

In [ ]:

df_train_id_sepsis = df_train.groupby(by="patientunitstayid")[["sepsis_patient"]]
df_train_id_sepsis = df_train_id_sepsis.first().reset_index()

non_sepsis_patient = df_train_id_sepsis[df_train_id_sepsis['sepsis_patient'] == False].iloc[2]
sepsis_patient = df_train_id_sepsis[df_train_id_sepsis['sepsis_patient'] == True].iloc[2]


In [ ]:
non_sepsis_patient

In [ ]:
sepsis_patient

In [ ]:
df_train_control_sample = df_train[df_train["patientunitstayid"] == non_sepsis_patient["patientunitstayid"]]
df_train_sepsis_sample = df_train[df_train["patientunitstayid"] == sepsis_patient["patientunitstayid"]]


In [ ]:
df_train_control_sample

In [ ]:
df_train_sepsis_sample

Due to splitting and rus almost no patient stays that are longer than one sample -> we need to retrieve it via the train dataset or val

In [ ]:
df_train["patientunitstayid"].value_counts()

In [ ]:
df_los_ids = df_train["patientunitstayid"].value_counts().reset_index()
df_los_ids

In [ ]:
df_long_ids = df_los_ids[df_los_ids["count"] > 144]
df_long_ids

In [ ]:
df_train[df_train["patientunitstayid"] == df_long_ids.iloc[0, 0]]

In [ ]:
df_long_ids

df_train_long_stays = df_train[df_train["patientunitstayid"].isin(df_train["patientunitstayid"].value_counts()["count"] )]
df_train_long_stays

In [ ]:
control_id = df_long_ids.iloc[0, 0]
sepsis_id = df_long_ids.iloc[2, 0]

df_long_control_sample = df_train[df_train["patientunitstayid"] == control_id].reset_index()
df_long_sepsis_sample = df_train[df_train["patientunitstayid"] == sepsis_id].reset_index()

In [ ]:
df_long_control_sample.head()

In [ ]:
df_long_sepsis_sample.head()

In [ ]:

df_long_control_sample[["sepsis_present", "sepsis_sample_shifted"]] = df_long_control_sample[
    ["sepsis_present", "sepsis_sample_shifted"]].astype(int)
df_long_sepsis_sample[["sepsis_present", "sepsis_sample_shifted"]] = df_long_sepsis_sample[
    ["sepsis_present", "sepsis_sample_shifted"]].astype(int)
df_long_control_sample["time_offset"] = df_long_control_sample["time_offset"] - df_long_control_sample[
    "time_offset"].min()

In [ ]:
df_long_control_sample["patientunitstayid"].unique()

In [ ]:
df_long_sepsis_sample["patientunitstayid"].unique()

In [ ]:
df_long_control_sample

In [ ]:
df_long_sepsis_sample

In [ ]:
df_long_control_sample[["heartrate", "respiration", "sepsis_sample_shifted"]].plot()

In [ ]:
df_long_sepsis_sample[["heartrate", "respiration", "sepsis_sample_shifted"]].plot()


In [ ]:
df_long_sepsis_sample["sample_idx_6_h"].unique()

In [ ]:

df_long_control_sample["sample_idx_6_h"].unique()


In [ ]:
sepsis_sample_1 = sepsis_pred_data.train_dataset[1886]
sepsis_sample_2 = sepsis_pred_data.train_dataset[1887]
sepsis_sample_3 = sepsis_pred_data.train_dataset[1888]

control_sample_1 = sepsis_pred_data.train_dataset[528]
control_sample_2 = sepsis_pred_data.train_dataset[529]
control_sample_3 = sepsis_pred_data.train_dataset[530]
control_sample_4 = sepsis_pred_data.train_dataset[531]


In [ ]:
sepsis_sample_1[0].shape

In [ ]:
sepsis_sample_1[1]

In [ ]:
# sample_ts = batch_features_ts[sample_index].unsqueeze(0)
# sample_stat = batch_features_stat[sample_index].unsqueeze(0)

model.to("cpu")
model.eval()
with torch.no_grad():
    pred_sepsis_1, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sepsis_sample_1[0].unsqueeze(0), sepsis_sample_1[1].unsqueeze(0), stage="test")

    pred_sepsis_2, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sepsis_sample_2[0].unsqueeze(0), sepsis_sample_2[1].unsqueeze(0), stage="test")

    pred_sepsis_3, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sepsis_sample_3[0].unsqueeze(0), sepsis_sample_3[1].unsqueeze(0), stage="test")

    pred_control_1, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        control_sample_1[0].unsqueeze(0), sepsis_sample_1[1].unsqueeze(0), stage="test")

    pred_control_2, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        control_sample_2[0].unsqueeze(0), sepsis_sample_2[1].unsqueeze(0), stage="test")

    pred_control_3, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        control_sample_3[0].unsqueeze(0), sepsis_sample_3[1].unsqueeze(0), stage="test")

    pred_control_4, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        control_sample_4[0].unsqueeze(0), sepsis_sample_3[1].unsqueeze(0), stage="test")


In [ ]:
prob_sepsis_1 = nn.functional.sigmoid(pred_sepsis_1)

In [ ]:
prob_sepsis_2 = nn.functional.sigmoid(pred_sepsis_2)

In [ ]:
prob_sepsis_3 = nn.functional.sigmoid(pred_sepsis_3)

In [ ]:
prob_control_1 = nn.functional.sigmoid(pred_control_1)
prob_control_2 = nn.functional.sigmoid(pred_control_2)
prob_control_3 = nn.functional.sigmoid(pred_control_3)
prob_control_4 = nn.functional.sigmoid(pred_control_4)
print(prob_control_1, prob_control_2, prob_control_3, prob_control_4)

In [ ]:
probabilities_control = [((idx + 1) * 72, val.squeeze(0, 1).numpy()) for idx, val in
                         enumerate([prob_control_1, prob_control_2, prob_control_3, prob_control_4])]

probabilities_sepsis = [((idx + 1) * 72, val.squeeze(0, 1).numpy()) for idx, val in
                        enumerate([prob_sepsis_1, prob_sepsis_2, prob_sepsis_3])]


In [ ]:
np.array(probabilities_control)

In [ ]:
df_control_probabilities = pd.DataFrame(probabilities_control, columns=["time_offset", "sepsis_in_6h_probability"])
df_control_probabilities["sepsis_in_6h_probability"] = df_control_probabilities["sepsis_in_6h_probability"].astype(
    "float")
# df_control_probabilities["time_offset"] = pd.to_timedelta(df_control_probabilities["time_offset"] * 5, unit="min")
df_control_probabilities

In [ ]:

df_sepsis_probabilities = pd.DataFrame(probabilities_sepsis, columns=["time_offset", "sepsis_in_6h_probability"])

df_sepsis_probabilities["sepsis_in_6h_probability"] = df_sepsis_probabilities["sepsis_in_6h_probability"].astype(
    "float")
# df_control_probabilities["time_offset"] = pd.to_timedelta(df_control_probabilities["time_offset"] * 5, unit="min")
df_sepsis_probabilities

In [ ]:

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)


In [ ]:
fig_control, axs_control, _, font_annots = figure_setup(3, 1, height_cm=15, num_colors=5)

# df_long_control_sample.plot(x="time_offset", y=["heartrate", "respiration"], ax=axs_control[0])
df_long_control_sample.plot(y=["heartrate", "respiration"], ax=axs_control[0])

df_long_control_sample.plot(y="sepsis_sample_shifted", ax=axs_control[1],
                            #marker="o", linestyle="None",
                            # linewidth=3
                            )

df_control_probabilities.plot(x="time_offset", y="sepsis_in_6h_probability", ax=axs_control[
    2], marker="X", linestyle="--")

for ax in axs_control:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax.xaxis.set_major_formatter(five_min_hour_formatter)
    ax.set_xlim(left=0)

axs_control[0].set_ylim(0, 1)
axs_control[1].set_ylim(-0.1, 1)

axs_control[2].set_xlabel("Time [h]")

axs_control[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.4),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_control[1].legend(
    ["Sepsis-3 in 6 h"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.4),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_control[2].legend(
    ["Predicted Probability of Sepsis-3 in 6 h"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.4),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()

In [ ]:
fig_sepsis, axs_sepsis, _, _ = figure_setup(3, 1, height_cm=15)

# df_long_control_sample.plot(x="time_offset", y=["heartrate", "respiration"], ax=axs_control[0])
df_long_sepsis_sample.plot(y=["heartrate", "respiration"], ax=axs_sepsis[0])

# df_long_sepsis_sample.plot( y="sepsis_sample_shifted", ax=axs_sepsis[1], marker="o", linestyle="None")
df_long_sepsis_sample.plot(y="sepsis_sample_shifted", ax=axs_sepsis[1])

df_sepsis_probabilities.plot(x="time_offset", y="sepsis_in_6h_probability", ax=axs_sepsis[
    2], marker="X", linestyle="--")

for ax in axs_sepsis:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax.xaxis.set_major_formatter(five_min_hour_formatter)
    ax.set_xlim(left=0)
    # ax.set_ylim(bottom=0)

plt.tight_layout()


In [ ]:
sofa_path = project_root / Path("eicu_code_dhm/intermediary_data/outcome_labels/sepsis/valid/sofa_data_1h.parquet")

In [ ]:
df_sofa = load_parquet_table(sofa_path, backend="numpy_nullable")

In [ ]:
df_sofa.head()

In [ ]:
df_sofa_control = df_sofa[df_sofa["patientunitstayid"] == control_id].reset_index()
df_sofa_sepsis = df_sofa[df_sofa["patientunitstayid"] == sepsis_id].reset_index()



In [ ]:
df_sofa_control = df_sofa_control.loc[df_sofa_control["labresultoffset"] > pd.to_timedelta(0)]
df_sofa_sepsis = df_sofa_sepsis.loc[df_sofa_sepsis["labresultoffset"] > pd.to_timedelta(0)]


In [ ]:

df_sofa_sepsis["labresultoffset_hours"] = df_sofa_sepsis["labresultoffset"] / 3600

In [ ]:
df_sofa_sepsis.head()

In [ ]:
df_sofa_control.head()

In [ ]:
df_sofa_control["sofa"]

In [ ]:
df_sofa_control.plot(x="labresultoffset", y="sofa")

In [ ]:
df_sofa_sepsis.plot(x="labresultoffset", y="sofa")

In [ ]:
fig_sepsis, axs_sepsis, _, _ = figure_setup(3, 1, height_cm=15)

# df_long_control_sample.plot(x="time_offset", y=["heartrate", "respiration"], ax=axs_control[0])
df_long_sepsis_sample.plot(y=["heartrate", "respiration"], ax=axs_sepsis[0])

# df_long_sepsis_sample.plot( y="sepsis_sample_shifted", ax=axs_sepsis[1], marker="o", linestyle="None")
df_long_sepsis_sample.plot(y="sepsis_sample_shifted", ax=axs_sepsis[2])

df_sepsis_probabilities.plot(x="time_offset", y="sepsis_in_6h_probability", ax=axs_sepsis[
    2], marker="X", linestyle="--")

df_sofa_sepsis.plot(x="labresultoffset_hours", y="sofa", ax=axs_sepsis[1])

for ax in axs_sepsis:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax.xaxis.set_major_formatter(five_min_hour_formatter)
    ax.set_xlim(left=0)
    # ax.set_ylim(bottom=0)

plt.tight_layout()


In [ ]:
all_data_path = project_root / Path("datasets/sepsis_type_a_cohort/aligned_all/2024-08-08/df_all_aligned.parquet")

In [ ]:
df_all_data = load_parquet_table(all_data_path, backend="numpy_nullable")

In [ ]:
df_all_data_control = df_all_data[df_all_data["patientunitstayid"] == control_id].reset_index(drop=True)
df_all_data_sepsis = df_all_data[df_all_data["patientunitstayid"] == sepsis_id].reset_index(drop=True)


In [ ]:
df_all_data_sepsis

In [ ]:
# zero_columns_control = df_all_data_control.loc[:, (df_all_data_control == 0).all()].columns.to_list()
# df_all_data_control_present = df_all_data_control.drop(columns=zero_columns_control)
# df_all_data_control_present = df_all_data_control_present.drop(columns=["index", "patientunitstayid"])

# df_all_data_control_present

In [ ]:
pd.options.plotting.backend = "matplotlib"
# pd.options.plotting.backend = "hvplot"


In [ ]:
#columns_to_drop = [col for col in df_all_data_control_present.columns if df_all_data_control_present[col].max() > 150]

# Drop the offending columns
# df = df_all_data_control_present.drop(columns=columns_to_drop)
# df = df_all_data_control_present.drop(columns=columns_to_drop)
df_all_data_control_present = df_all_data_control[[*sepsis_pred_data.considered_features, "time_offset"]]
df_all_data_sepsis_present = df_all_data_sepsis[[*sepsis_pred_data.considered_features, "time_offset"]]


In [ ]:
def generate_sliding_windows(df, time_offset_col, window_size=72, step_size=1):
    """
    Generates sliding windows over the DataFrame based on the time_offset column.

    Parameters:
        df (pd.DataFrame): The input DataFrame with a time_offset column.
        time_offset_col (str): The name of the column representing time offset.
        window_size (int): The number of index steps corresponding to 6 hours of data.
        step_size (int): The number of index steps for each slide of the window (e.g., 5 minutes corresponds to step_size=1).

    Returns:
        list: A list of DataFrame windows.
    """

    sliding_windows = []

    df = df.sort_values(by=time_offset_col).reset_index(drop=True)

    start_index = window_size
    # start_index = 0 

    for i in range(start_index, len(df), step_size):
        window = df.iloc[i - window_size:i].copy()

        sliding_windows.append(window)

    return sliding_windows

In [ ]:
df_all_data_control_present = df_all_data_control_present.replace(to_replace=np.nan, value=float(0))
df_all_data_control_sepsis = df_all_data_sepsis_present.replace(to_replace=np.nan, value=float(0))


In [ ]:
import pickle

with open(project_root / "scaler_train.pkl", "rb") as f:
    scaler = pickle.load(f)

scaler_feature_names = scaler.get_feature_names_out()
scaler_feature_names

In [ ]:
df_all_data_control_present = df_all_data_control_present[["time_offset", *scaler_feature_names]]

df_all_data_control_present.loc[:, sepsis_pred_data.considered_features] = scaler.transform(
    df_all_data_control_present.loc[:, scaler_feature_names])

df_all_data_sepsis_present = df_all_data_sepsis_present[["time_offset", *scaler_feature_names]]

df_all_data_sepsis_present.loc[:, sepsis_pred_data.considered_features] = scaler.transform(
    df_all_data_sepsis_present.loc[:, scaler_feature_names])


In [ ]:
df_all_data_sepsis_present

sliding_windows = []

df = df_all_data_control_present.sort_values(by="time_offset").reset_index(drop=True)

start_index = 72
step_size = 1
window_size = 72

for i in range(start_index, len(df), step_size):
    window = df.iloc[i - window_size:i].copy()

    sliding_windows.append(window)
    


len(sliding_windows)
    

In [ ]:
windows_control = generate_sliding_windows(df=df_all_data_control_present, time_offset_col="time_offset",
                                           window_size=72, step_size=1)

windows_sepsis = generate_sliding_windows(df=df_all_data_control_sepsis, time_offset_col="time_offset",
                                          window_size=72, step_size=1)


In [ ]:
def stack_windows_as_tensor(sliding_windows, feature_columns):
    """
    Stacks the sliding windows into a 3D tensor for model input.

    Parameters:
        sliding_windows (list of pd.DataFrame): The list of DataFrame windows.
        feature_columns (list of str): The list of columns to include in the tensor (excluding time offset).

    Returns:
        np.ndarray: A 3D tensor of shape (batch_size, sequence_length, num_features).
    """

    # Convert each DataFrame window to a NumPy array and stack them
    tensor = np.stack([window[feature_columns].to_numpy() for window in sliding_windows])

    return tensor

In [ ]:
incremental_step_window_tensor_control = stack_windows_as_tensor(windows_control,
                                                                 feature_columns=sepsis_pred_data.considered_features)

incremental_step_window_tensor_sepsis = stack_windows_as_tensor(windows_sepsis,
                                                                feature_columns=sepsis_pred_data.considered_features)


In [ ]:
incremental_step_window_tensor_control.shape

In [ ]:
incremental_step_window_tensor_sepsis.shape

In [ ]:
incremental_step_window_tensor_control = incremental_step_window_tensor_control.astype(np.float32)
incremental_step_window_tensor_sepsis = incremental_step_window_tensor_sepsis.astype(np.float32)


In [ ]:

data_tensor_control = torch.from_numpy(incremental_step_window_tensor_control).to("cpu")
data_tensor_sepsis = torch.from_numpy(incremental_step_window_tensor_sepsis).to("cpu")

df_train_static = sepsis_pred_data.df_static_train_processed

control_features_static = [col for col in df_train_static.columns if col != "patientunitstayid"]

df_static_control = df_train_static[df_train_static["patientunitstayid"] == control_id].reset_index(drop=True)
df_static_control = df_static_control[control_features_static]

df_static_sepsis = df_train_static[df_train_static["patientunitstayid"] == sepsis_id].reset_index(drop=True)
df_static_sepsis = df_static_sepsis[control_features_static]
df_static_sepsis.head()

In [ ]:
# static_in_control_windows = torch.tile(torch.from_numpy(df_static_control.to_numpy(dtype=np.float32)), (incremental_step_window_tensor_control.shape[0], 1, 1))
# static_in_control_windows

In [ ]:
data_tensor_control.shape

In [ ]:
batch_input_data_ts_control = data_tensor_control
batch_input_data_static_control = torch.tile(torch.from_numpy(df_static_control.to_numpy(dtype=np.float32)),
                                             (data_tensor_control.shape[0], 1, 1)).squeeze(1)
batch_input_data_static_control.shape

In [ ]:
batch_input_data_ts_sepsis = data_tensor_sepsis
batch_input_data_static_sepsis = torch.tile(torch.from_numpy(df_static_sepsis.to_numpy(dtype=np.float32)),
                                            (data_tensor_sepsis.shape[0], 1, 1)).squeeze(1)
batch_input_data_static_sepsis.shape

batch inference (plenty fast)


In [ ]:
from tqdm.auto import tqdm

model.eval()
model.to("cuda")

# predictions_control = []

with torch.no_grad():
    # Perform forward pass to get predictions
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        batch_input_data_ts_control.to("cuda"), batch_input_data_static_control.to("cuda"), stage="test")

    # predictions_control.append(pred.cpu().numpy())
    preds_control = torch.nn.functional.sigmoid(pred).to("cpu").numpy()

    # predictions = np.concatenate(predictions_control, axis=0)
    predictions_control = np.concatenate(preds_control, axis=0)




In [ ]:
with torch.no_grad():
    # Perform forward pass to get predictions
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        batch_input_data_ts_sepsis.to("cuda"), batch_input_data_static_sepsis.to("cuda"), stage="test")

    # predictions_control.append(pred.cpu().numpy())
    preds_sepsis = torch.nn.functional.sigmoid(pred).to("cpu").numpy()

    # predictions = np.concatenate(predictions_control, axis=0)

    predictions_sepsis = np.concatenate(preds_sepsis, axis=0)




In [ ]:
df_all_data_control_present["time_offset_hours"] = df_all_data_control_present["time_offset"].dt.total_seconds() / 3600
df_all_data_sepsis_present["time_offset_hours"] = df_all_data_sepsis_present["time_offset"].dt.total_seconds() / 3600

df_all_data_control_present["prediction"] = np.nan
df_all_data_sepsis_present["prediction"] = np.nan

df_all_data_control_present.loc[72:, "prediction"] = predictions_control
df_all_data_sepsis_present.loc[72:, "prediction"] = predictions_sepsis

df_all_data_control_present

In [ ]:
df_sofa_control

In [ ]:
df_all_data_sepsis_present

In [ ]:
df_sofa_control["time_offset_hours"] = df_sofa_control["labresultoffset"].dt.total_seconds() / 3600
df_sofa_sepsis["time_offset_hours"] = df_sofa_sepsis["labresultoffset"].dt.total_seconds() / 3600

# df_all_data_sepsis["time_offset_hours"] = df_all_data_sepsis["time_offset"].dt.total_seconds() / 3600

# df_sofa_control["time_offset_hours"] = df_sofa_control["time_offset_hours"].astype(int)
# df_sofa_sepsis["time_offset_hours"] = df_sofa_sepsis["time_offset_hours"].astype(int)

# df_all_data_sepsis["time_offset_hours"] =  df_all_data_sepsis["time_offset_hours"].astype(int)

# limiting the sofa data to available data

df_sofa_control = df_sofa_control.loc[
    df_sofa_control["labresultoffset"] <= df_all_data_control_present["time_offset"].max()]
df_sofa_sepsis = df_sofa_sepsis.loc[
    df_sofa_sepsis["labresultoffset"] <= df_all_data_sepsis_present["time_offset"].max()]
df_sofa_sepsis

In [ ]:

lab_names = VariableNames.get_lab_related_names()
lab_names_original = list(lab_names.keys())
lab_names_pretty = list(lab_names.values())

vital_names = VariableNames.get_vital_sign_names()
vital_names_original = list(vital_names.keys())
vital_names_pretty = list(vital_names.values())
vital_names_pretty

In [ ]:
df_all_data_control_present = VariableNames.get_pretty_dataframe(df_all_data_control_present)
df_all_data_sepsis_present = VariableNames.get_pretty_dataframe(df_all_data_sepsis_present)


In [ ]:
df_all_data_control_present.head()

In [ ]:
# df_all_data_control_present = df_all_data_control_present.replace(0.0, pd.NA)
# df_all_data_sepsis_present = df_all_data_sepsis_present.replace(0.0, pd.NA)



In [ ]:
from machine_intelligence.utilities.viz_setup import VariableNames

fig_control, axs_control, _, font_annots = figure_setup(2, 1, height_cm=30, num_colors=20)

# dense ts variables
df_all_data_control_present.plot(x="time_offset_hours", y=vital_names_pretty, ax=axs_control[0])
# sparse ts variables
df_all_data_control_present.plot(x="time_offset_hours", y=lab_names_pretty, ax=axs_control[1], marker="x",
                                 linestyle=None)

for ax in axs_control:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 24))
    ax.set_xlim(left=0, right=df_all_data_control_present["time_offset_hours"].max())

axs_control[0].set_ylim(0, 1)
# axs_control[1].set_ylim(-0.1, 1)
# axs_control[2].set_xlabel("Time [h]")


axs_control[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_control[1].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()



In [ ]:
fig_control_2, axs_control_2, _, font_annots = figure_setup(2, 1, height_cm=15, num_colors=5, color_offset=2,
                                                            share_x=True)

df_sofa_control.plot(x="time_offset_hours", y="sofa", ax=axs_control_2[0],
                     linewidth=2,
                     # marker="x"
                     )

df_all_data_control_present.plot(x="time_offset_hours", y="prediction", ax=axs_control_2[
    1],
                                 )

axs_control_2[0].yaxis.set_major_locator(plt.MultipleLocator(2))

axs_control_2[1].set_xlabel("Time [h]")

axs_control_2[0].legend(
    ["SOFA Score"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_control_2[1].legend(
    ["Predicted Probability of Sepsis-3 in 6 h"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()


In [ ]:
import matplotlib as mpl


def plot_sparse_with_random_markers(ax, x, y, marker_list):
    for col in y.columns:
        # Randomly selecting marker positions
        mask = np.random.choice([True, False], size=len(x), p=[0.3, 0.7])  # Adjust p for marker frequency
        # ax.plot(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7)
        jittered_x = x[mask] + np.random.normal(0, 0.3, size=len(x[mask]))  # Adjust 0.1 for more or less jitter
        ax.scatter(jittered_x, y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, s=10,
                   alpha=0.7)
        # ax.scatter(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7, s=10)


# Generate the figure and axes
fig_control, axs_control, _, font_annots = figure_setup(2, 1, height_cm=30, num_colors=19)

# dense ts variables
# df_all_data_control_present.plot(x="time_offset_hours", y=vital_names_pretty, ax=axs_control[0],
#                                  colormap="tab20"
#                                  )


line_styles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--", "-.", ":", ]

# Plot dense ts variables with different line styles
for i, col in enumerate(vital_names_pretty):
    axs_control[0].plot(df_all_data_control_present["time_offset_hours"],
                        df_all_data_control_present[col],
                        linestyle=line_styles[i % len(line_styles)],
                        label=col)

# sparse ts variables with random markers
plot_sparse_with_random_markers(axs_control[1],
                                df_all_data_control_present["time_offset_hours"],
                                df_all_data_control_present[lab_names_pretty].replace(0.0, pd.NA),
                                marker_list=['x', 'o', 's', 'D', '*'])

for ax in axs_control:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 24))
    ax.set_xlim(left=0, right=df_all_data_control_present["time_offset_hours"].max())

axs_control[0].set_ylim(0, 1)

# Customizing legends
axs_control[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_control[1].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()

In [ ]:

def plot_sparse_with_random_markers(ax, x, y, marker_list):
    for col in y.columns:
        # Randomly selecting marker positions
        mask = np.random.choice([True, False], size=len(x), p=[0.3, 0.7])  # Adjust p for marker frequency
        # ax.plot(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7)
        jittered_x = x[mask] + np.random.normal(0, 0.3, size=len(x[mask]))  # Adjust 0.1 for more or less jitter
        ax.scatter(jittered_x, y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, s=10,
                   alpha=0.7)
        # ax.scatter(x[mask], y[col][mask], marker=np.random.choice(marker_list), linestyle='', label=col, linewidth=0.5, alpha=0.7, s=10)


# Generate the figure and axes
fig_sepsis, axs, _, font_annots = figure_setup(2, 1, height_cm=25, num_colors=20,
                                               # palette=palettable.scientific.sequential.Hawaii_20.mpl_colors,
                                               palette=palettable.matplotlib.Plasma_20.mpl_colors[:18],
                                               )

# dense ts variables
# df_all_data_control_present.plot(x="time_offset_hours", y=vital_names_pretty, ax=axs_control[0],
#                                  colormap="tab20"
#                                  )


# line_styles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--", "-.", ":", ]
line_styles = ["-", "--", "-.", ":", ]
line_styles =  line_styles  + line_styles[::-1]


# Plot dense ts variables with different line styles
for i, col in enumerate(vital_names_pretty):
    axs[0].plot(df_all_data_sepsis_present["time_offset_hours"],
                df_all_data_sepsis_present[col],
                linestyle=line_styles[i % len(line_styles)],
                label=col)

# sparse ts variables with random markers
plot_sparse_with_random_markers(axs[1],
                                df_all_data_sepsis_present["time_offset_hours"],
                                df_all_data_sepsis_present[lab_names_pretty].replace(0.0, pd.NA),
                                marker_list=['x', 'o', 's', 'D', '*', "h", "+", "p", "2", "8"])

for ax in axs:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 24))
    ax.set_xlim(left=0, right=df_all_data_sepsis_present["time_offset_hours"].max())

axs[0].set_ylim(0, 1)


axs[0].set_ylabel("Dense Time Series Features")
axs[1].set_ylabel("Sparse Time Series Features")

axs[1].set_xlabel("Time [h]")

font_annots.set_size(6)  # Set the desired font size
# Customizing legends
axs[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs[1].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()


if save_as_svg := True:
    fig_sepsis.savefig(plot_output_dir / Path("patient_risk_plot_variables.svg"), format="svg")



In [ ]:
df_all_data_sepsis["sepsis_present"] = df_all_data_sepsis["sepsis_present"].astype(int)

df_all_data_sepsis["time_offset_hours"] = df_all_data_sepsis["time_offset"].dt.total_seconds() / 3600
df_all_data_sepsis["time_offset_hours"] = df_all_data_sepsis["time_offset_hours"]

df_all_data_sepsis.head()

In [ ]:
df_all_data_sepsis_present.head()

In [ ]:
df_sofa_sepsis.head()

In [ ]:
fig_sepsis_2, axs_2, _, font_annots = figure_setup(2, 1, height_cm=7, num_colors=5, color_offset=1, share_x=True)

df_sofa_sepsis.plot(x="time_offset_hours", y="sofa", ax=axs_2[0],
                    linewidth=2,
                    )

df_all_data_sepsis_present.plot(x="time_offset_hours", y="prediction", ax=axs_2[
    1],
                                )

df_all_data_sepsis.plot(x="time_offset_hours", y="sepsis_present", ax=axs_2[
    1],
    linewidth=2,
                        )

axs_2[0].yaxis.set_major_locator(plt.MultipleLocator(2))

axs_2[0].xaxis.set_major_locator(plt.MultipleLocator(1 * 24))
axs_2[1].xaxis.set_major_locator(plt.MultipleLocator(1 * 24))
axs_2[0].set_xlim(left=0, right=df_all_data_sepsis_present["time_offset_hours"].max())
axs_2[1].set_xlim(left=0, right=df_all_data_sepsis_present["time_offset_hours"].max())

axs_2[0].minorticks_off()
axs_2[1].minorticks_off()

# axs_2[1].set_ylim(0, 1)

axs_2[0].set_ylabel("SOFA Score")
axs_2[1].set_ylabel("Probability")

axs_2[1].set_xlabel("Time [h]")

axs_2[0].legend(
    ["SOFA"],
    # loc="upper center",
    loc="best",
    # bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    # frameon=False,
    frameon=True,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

axs_2[1].legend(
    ["Sepsis-3 Prediction", "Sepsis-3 Label"],
    # loc="upper center",
    loc="best",
    # bbox_to_anchor=(0.5, -0.35),
    ncol=3,
    # frameon=False,
    frameon=True,
    fancybox=True,
    shadow=True,
    prop=font_annots,
)

plt.tight_layout()



if save_as_svg := True:
    fig_sepsis_2.savefig(plot_output_dir / Path("patient_risk_plot_pred_7cm.svg"), format="svg")


### Note: The data being plotted is different every time the datamodule is setup, we get the sample from the training set and that gets shuffled, so this is expected behaviour